# 成果：能够自我验证的智能体

智能体擅长产出*看起来*已经完成的东西。向它要一份带引用的研究简报，你会得到一份脚注整齐的文档。凑近看，通常还有改进空间：某个主题写得单薄，某处引文偏离了原文，某条引用依赖的是新闻稿而不是原始申报文件。要发现这些问题需要人工复核循环：你读输出、找出不对的地方、再重新提示。而你在这些轮次里说的内容，大多是可以在智能体开始之前就写下来的反馈。

Claude Managed Agents 中的[成果（Outcomes）](https://platform.claude.com/docs/en/managed-agents/define-outcome)为会话配备第二个智能体，它唯一的职责就是检查。你写一份**评分细则（rubric）**，说明“完成”是什么样子以及如何验证，平台会在它自己的上下文窗口中配置一个**评分器（grader）**。评分器看不到写作者的推理，也不知道走了哪些捷径。每次写作者轮次结束后，它会对照评分细则重新读取产物，要么通过，要么交回一份逐条标准的缺口清单。写作者据此修改，循环再次运行，直到你设定的上限。

在本指南中，你会看到这个循环端到端地运行。写作者起草一份关于电动汽车快充经济性的一页简报，并逐字引用它的来源。评分器独立抓取每一条被引用的 URL，在每个页面上搜索被引用的字符串，核实引文确实支持该论断，并对照七项清单为覆盖度打分。你会看到它抓到一个真实问题（该引用 10-K 却引用了新闻稿附件），也会看到写作者去找到正确的文件。

## 你将学到什么

- 写一份评分器可据此执行的评分细则
- 带着一个成果开始会话，让智能体朝它推进
- 跟踪评分与修改的循环，并读懂评分器的反馈
- 判断什么时候成果是合适的工具


## 1. 准备环境

首先安装 SDK 并配置 Anthropic 客户端。`define_outcome` 事件和成果评估跨度类型属于 Managed Agents 测试版的一部分。


In [1]:
%%capture
%pip install -q anthropic python-dotenv

In [2]:
import os
import re
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = anthropic.Anthropic()

## 2. 创建写作者并开启会话

接下来创建写作者智能体并开启会话。写作者的系统提示词要求它引用不超过六个来源，每个来源配一段简短的逐字引文，这样评分器才有可核查的具体内容。


In [3]:
env = client.beta.environments.create(
    name="research-brief",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
)

writer = client.beta.agents.create(
    name="Research Analyst",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-verify-with-outcome-grader"},
    model=MODEL,
    system="""You are a research analyst. You write one-page business briefs.

Cite every factual claim with an inline footnote [n]. End the brief with a Sources section in this exact format, one entry per line:

[n] "verbatim quote from the page, 25 words or fewer" - Title - URL

Only cite pages you actually fetched and read. The quote must be copied character-for-character from the page. Cite no more than 6 sources total. Pick the strongest; do not pad. Save the brief to /mnt/session/outputs/brief.md.""",
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [
                {"name": "web_search"},
                {"name": "web_fetch"},
                {"name": "read"},
                {"name": "write"},
            ],
        }
    ],
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": writer.id, "version": writer.version},
    environment_id=env.id,
    title="Brief: EV fast-charging unit economics",
    betas=BETAS,
)
print(f"Session {session.id}")

Session sesn_011CakRYQjc4NMXdRy4qKZy5


## 3. 写一份评分器可据此执行的评分细则

`define_outcome` 事件交给会话两样东西：交付物的**描述（description）**，由写作者阅读；以及**评分细则（rubric）**，由评分器阅读。每次写作者轮次结束后，平台会启动一个新的评分器，使用与写作者相同的模型和工具，把评分细则交给它，并让它检查产物。评分器返回逐条标准的判定，如果有任何一项不通过，说明就直接交回写作者用于下一次修改。

评分细则是你对评分器唯一的抓手，它的措辞决定评分器是否真的检查了任何东西。默认的失效模式是一个什么都批准的评分器。写着*“检查简报是否涵盖需量电费”*的评分细则，评分器可以扫一眼简报，看到一段讲需量电费的内容，然后写一句确认它在那里。它可以完全不打开任何来源就完成这一切。多数初稿凭这一点就能通过，循环永远不会运行。而写着*“打开简报，找到需量电费那一节，确认它给出了 $/kW 的数值或占运营成本的百分比”*的评分细则，会迫使评分器拿出证据。正是这种评分器能抓出冒充 10-K 的新闻稿。

每份评分细则都值得做这几件事：

| 原则 | 实际做法 |
|---|---|
| **让每条标准都可核查** | 任务说的是“具名运营商的经济性”。评分细则把它钉死：来自 10-K 或 10-Q 的 GAAP 净亏损，并引用 `sec.gov` 上的申报文件。评分细则应始终比任务更具体。|
| **让评分器挣得 `satisfied`** | 在评分器通过任何内容之前，要求具体证据（抓取到的页面、追溯过的公式、`file:line` 引用）。过于严格的评分器会让你多跑一轮。过于宽松的评分器会在糟糕版本仍在原位时结束循环。|
| **描述目标，而不是步骤** | 规定某个具体命令的评分细则，在该命令不可用时会静默失败，你在意的那项检查永远不会运行。定义什么算作证据；评分器拥有写作者的完整工具集，能够自己找到。|
| **预判写作者的捷径** | “不得通过镜像、转载页或搜索摘要来佐证。”没有这一句，失效来源会被换成一个抓取页，评分器就放过了。|
| **规定反馈的格式** | 评分器的说明是写作者唯一的信号。要求先给一行记分板，然后每个不通过项一条要点，说明问题在哪、该怎么做。|
| **告诉评分器该忽略什么** | 没有不予触发清单，评分器会在风格小问题、既有问题和范围扩张上瞎折腾。把越界内容写清楚，并要求它在提出每条发现之前先自检。|

> **还没有评分细则？** 把一份公认优秀的产物示例交给 Claude，让它分析这份示例好在哪里，再把这份分析转成标准。这个中间做法稳定胜过从空白页直接写标准。

下面是我们这份简报的任务和评分细则。把它们并排读一遍，注意评分细则具体得多。它每一处比任务多出的细节，都是评分器真正能跑的一项检查。


In [4]:
TASK = """Write a brief on the unit economics of public DC fast charging in the United States.
The brief should cover:
  1. Capex range
  2. Demand charges
  3. Utilization breakeven
  4. Subsidy programs
  5. Named-operator economics
  6. A contrarian or skeptical source
  7. Hardware vs install cost split
"""


RUBRIC = """
You are reviewing a research brief at /mnt/session/outputs/brief.md against a coverage checklist and verifying its citations. The writer was told the seven topics to cover; this rubric defines what counts as sufficient coverage for each topic, and how to verify citations.

COVERAGE CHECKLIST. Each item has a specific area:
  1. Capex range: a dollar range for installed cost per DC fast-charging stall or station.
  2. Demand charges: quantified impact on opex (a $/kW figure or a % of operating cost).
  3. Utilization breakeven: a breakeven or target utilization threshold (% or kWh/day).
  4. Subsidy programs: NEVI or another public funding program, named.
  5. Named operator: the GAAP net income or net loss from a specific public charging operator's most recent 10-K or 10-Q, and the citation for it must be the SEC filing itself (sec.gov), not a press release, earnings-call recap, or news article.
  6. Contrarian source: at least one cited source whose thesis is that the economics are unfavorable or structurally challenged.
  7. Cost split: a hardware vs soft-cost (install, permitting, grid) breakdown or ratio.

CITATION CHECK. For every [n] entry in the Sources section:
  a. LIVE: Fetch the URL with web_fetch. Mark LIVE only if web_fetch returns the readable page directly. Mark DEAD if 404, parked, login-walled, paywalled, returns a bot-block/403, or renders only via JavaScript. Do NOT corroborate via mirrors, reposts, or search snippets; the cited URL itself must fetch.
  b. VERBATIM: Search the fetched page for the quoted string. Mark QUOTE_MATCH if the exact string appears (treat curly vs straight quotes as equivalent); NOT_FOUND otherwise.
  c. SUPPORTS CLAIM: Mark SUPPORTS_CLAIM if the quoted passage actually backs the claim it's cited on in the brief; UNSUPPORTED if it's tangential, contradicts the claim, or is just a general statement of fact.

OUTPUT FORMAT:

Line 1: Coverage N/7. Citations M/K verified.

Then, for each failed item in the coverage checklist, create a new bullet, name the item and say what specific bar it failed in one sentence max per bullet. For example: "Item 3 Utilization breakeven - MISSING. <what's missing>".

Then, for each failed citation, create a new bullet with the format: "[n] domain - REASON. <what's wrong and what to do>". One sentence max per bullet. For example: "[3] evgo.com - DEAD. The URL returns a 403 error and appears to be behind a bot block. No mirrors or reposts; the cited URL itself must fetch."
"""

client.beta.sessions.events.send(
    session.id,
    betas=BETAS,
    events=[
        {
            "type": "user.define_outcome",
            "description": TASK,
            "rubric": {"type": "text", "content": RUBRIC},
            "max_iterations": 5,
        },
    ],
)

BetaManagedAgentsSendSessionEvents(data=[BetaManagedAgentsUserDefineOutcomeEvent(id='sevt_011CakRYTKNkXJN7eh4kRs1r', description='Write a brief on the unit economics of public DC fast charging in the United States.\nThe brief should cover:\n  1. Capex range\n  2. Demand charges\n  3. Utilization breakeven\n  4. Subsidy programs\n  5. Named-operator economics\n  6. A contrarian or skeptical source\n  7. Hardware vs install cost split\n', max_iterations=5, outcome_id='outc_011CakRYTKNkJ83CBexcz13e', processed_at=datetime.datetime(2026, 5, 6, 1, 10, 30, 512206, tzinfo=TzInfo(0)), rubric=BetaManagedAgentsTextRubric(content='\nYou are reviewing a research brief at /mnt/session/outputs/brief.md against a coverage checklist and verifying its citations. The writer was told the seven topics to cover; this rubric defines what counts as sufficient coverage for each topic, and how to verify citations.\n\nCOVERAGE CHECKLIST. Each item has a specific area:\n  1. Capex range: a dollar range for insta

关于上面这次调用，有几点说明：

- **描述和评分细则各有职责。** 描述告诉写作者做什么。评分细则告诉评分器如何检查它。两者应当就产物的位置和格式达成一致；如果它们互相矛盾（比如描述要求内联输出，而评分细则给 `/mnt/session/outputs/` 下的文件打分），循环会返回 `failed`，而不是空转。
- **`max_iterations` 默认是 3（最大 20）。** 我们设为 5，因为这份评分细则严格，写作者需要空间。如果每次运行都撞上上限，而评分器每次都发现同一类问题，说明写作者无法据反馈行动，你在为不收敛的轮次付费。
- **内联评分细则与文件评分细则。** 在笔记本里用内联文本没问题。生产环境会通过 Files API 上传一次评分细则，并传 `rubric: {"type": "file", "file_id": ...}`，这样它可跨会话复用，并且能像代码一样接受评审。

**为什么不直接把评分细则放进系统提示词？** 可以，而且它确实帮助写作者把目标定得更准。但知道标准的写作者仍然是在给自己的作品打分。它会在自己认为通过时声称通过，也不会回头重新抓取它已经引用过的 URL，或注意到它记忆中的引文与页面上的引文略有差异。评分器则别无选择，只能做这些检查。它以全新的上下文窗口开始，除了评分细则和产物一无所有，而平台不允许循环继续，直到它对每一条标准都给出判定。无论你把单个提示词写得多好，都得不到这种分离。


## 4. 观察复核循环

我们来流式读取事件，并在每个阶段发生时呈现它。写作者完成一份草稿时打印一条横幅，每次评估后展示评分器的反馈。


下面这个循环用到两个显示辅助函数：`banner` 画一条带标签的分隔线，`render_feedback` 去掉服务端在每条评分器说明外面包裹的模板文字。


In [5]:
HR = "━" * 46


def banner(label, tag=""):
    print(f"\n{HR}\n{label}  {tag}".rstrip())


def render_feedback(fb: str):
    # Strip the server's per-criterion wrapper and trailer.
    s = re.sub(
        r"^An independent grader found.*?:\n\n- .*?\((?:partially |not )?met\): ",
        "",
        fb,
        count=1,
        flags=re.S,
    )
    s = re.sub(r"\n\nPlease revise your work.*$", "", s, flags=re.S)
    print(s)

In [6]:
TERMINAL = {"satisfied", "max_iterations_reached", "failed", "interrupted"}
t0, res, iters = time.time(), None, 0
n_search, last_len = 0, 0

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    for ev in stream:
        if ev.type == "agent.tool_use":
            if ev.name in ("web_search", "web_fetch"):
                n_search += 1
            if ev.name == "write" and ev.input["file_path"].endswith("brief.md"):
                last_len = len(ev.input["content"])
        elif ev.type == "span.outcome_evaluation_start":
            banner("writer · " + ("draft" if iters == 0 else f"revision {iters}"))
            print(f"searched/fetched {n_search}× · wrote brief.md ({last_len:,} chars)")
            n_search = 0
        elif ev.type == "span.outcome_evaluation_end":
            res = ev.result
            banner(
                f"grader · pass {iters}",
                "✓ satisfied" if res == "satisfied" else "⟳ needs_revision",
            )
            render_feedback(ev.explanation)
            iters += 1
            if res in TERMINAL:
                break

m, s = divmod(int(time.time() - t0), 60)
print(f"\ndone: {res} after {iters} pass{'es' if iters != 1 else ''} · {m}m {s:02d}s")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
writer · draft
searched/fetched 18× · wrote brief.md (7,607 chars)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
grader · pass 0  ⟳ needs_revision
The brief covers 5 of 7 required topics adequately. Item 2 (Demand charges) fails the quantification bar: the brief describes demand charges qualitatively as the 'single largest operational wildcard' but never states a $/kW figure (McKinsey's footnote of $20/kW is never quoted in the text) or a % of operating cost. Item 5 (Named operator) fails the citation requirement: EVgo's Q1 2024 net loss of $28.2M is cited to evchargingstations.com [6], a third-party news article, not an SEC filing from sec.gov as the rubric explicitly requires. All 6 citations are LIVE, and all 6 quoted strings match the fetched pages and support the claims they are attached to — but [6] is the wrong source type for the Named Operator criterion.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
writer · revision 1
search

### 刚才发生了什么

循环跑了三次评分。

第一份草稿覆盖了七项中的五项。评分器把它退回，因为两处缺失：需量电费只是定性描述，没有 $/kW 的金额；具名运营商那一节引用的是第三方新闻文章，而不是 SEC 申报文件。写作者补上了 $20/kW 这个数字，并到 sec.gov 去查 EVgo 的 FY2024 净亏损。

第二次评分通过七项中的六项。写作者引用的 sec.gov 文件是 8-K 的 Exhibit 99.1，即按申报形式提交的业绩新闻稿，不是评分细则所要求的 10-K 或 10-Q。评分器读了这个 URL，把它识别为新闻稿附件，并恰恰基于这一区分予以否决。第三次评分时，写作者在 EDGAR 上找到 EVgo 真正的 10-K，评分器放过了它：覆盖度 7/7，引用 6/6 全部可访问，引文匹配且支持论断。

两次否决都发生是因为评分细则画出了一条评分器能够核查的界线。“$/kW 的数值或占运营成本的百分比”和“SEC 申报文件本身，而不是新闻稿”，是评分细则作者做出的决定；任务本身（“涵盖需量电费”、“具名运营商的经济性”）会把这两处都放行。第二次否决值得研究。写作者找到了一个 `sec.gov` 的 URL，表面上满足任务，而评分器仍然把它打回，因为评分细则区分了 8-K 的新闻稿附件和 10-K。没有这一区分，循环会提前一轮结束，留下较差的简报。

**说明** 上面的轨迹来自最初的 cookbook 运行。如果你自己运行，结果很可能不同。

## 5. 阅读最终简报

最后，取回写作者产出的 `brief.md` 最后一个版本，看看它覆盖了什么、引用了什么。想读完整正文就直接打印 `content`。

In [7]:
# Reconstruct the final brief from the event log: full content on `write`,
# then apply each `edit` (old_string -> new_string) in order.
content = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if ev.type != "agent.tool_use" or "brief.md" not in str(ev.input.get("file_path", "")):
        continue
    if ev.name == "write":
        content = ev.input["content"]
    elif ev.name == "edit":
        content = content.replace(ev.input["old_string"], ev.input["new_string"], 1)

# Show the structure and sources rather than the full prose.
for line in content.splitlines():
    if line.startswith(("#", "[")):
        print(line)

# Unit Economics of Public DC Fast Charging in the United States
## 1. Capex Range
## 2. Demand Charges
## 3. Utilization Breakeven
## 4. Subsidy Programs
## 5. Named-Operator Economics
## 6. Contrarian / Skeptical View
## 7. Hardware vs. Installation Cost Split
## Sources
[1] "A 150 to 350kW DCFC charging unit can cost anywhere from $45,000 to over $100,000, and installation costs can range from $40,000 to over $150,000." - Can public EV fast-charging stations be profitable in the United States? - https://www.mckinsey.com/features/mckinsey-center-for-future-mobility/our-insights/can-public-ev-fast-charging-stations-be-profitable-in-the-united-states
[2] "Waiving demand charges for fast chargers will shift costs to all ratepayers, including non-EV drivers." - Fast charging, high costs: Eliminating demand charges won't solve the problem - https://www.utilitydive.com/news/eliminating-demand-charges-wont-solve-EV-station-problems/689395/
[3] "the average total NEVI project cost is $915,42

## 你学到了什么

- **成果适用于你能写下“好”是什么样子的时候。** 对细节的关注和穷尽式覆盖这类任务最契合；主观质量也可以，只要评分细则把标准钉下来。
- **评分细则必须让评分器产出证据。** 没有这一点，它会批准它看到的任何东西。
- **评分器独立且无状态。** 它在自己的上下文窗口中运行，所以写作者无法说服它，而且每一轮都会用一个新的评分器重新检查整个产物。
- **一次一个成果，但可以串联。** 循环一旦终止，会话恢复为对话式，下一次 `user.define_outcome` 会开启新的循环。

更多内容参见[成果文档](https://platform.claude.com/docs/en/managed-agents/define-outcomes)。
